# 面试问题：怎样实现多租户 LoRA Adapter 服务，并安全复用显存池？

**一句话回答。** 每个请求必须绑定 base checkpoint、adapter id/version、rank/shape 和租户授权；调度器只合并兼容请求，内存池只能驱逐未被引用的 adapter，更新走新版本而不能原地覆盖 in-flight 请求。

本 Notebook 仅用 Python 标准库手写数据合同、核心算法和失败分支；受控样例用于验证不变量，不代表生产吞吐、模型质量或硬件精度。

**资料入口。** [S-LoRA](https://arxiv.org/abs/2311.03285) 讨论多并发 LoRA adapter 的统一分页；本例用纯 Python 演示正确性合同而非 GPU kernel。


In [ ]:
question = "多 LoRA Adapter 服务"  # 执行本行的状态、计算或校验逻辑。
assert "LoRA" in question  # 执行本行的状态、计算或校验逻辑。
assert 2 < 4  # 执行本行的状态、计算或校验逻辑。
assert True  # 执行本行的状态、计算或校验逻辑。

## 1. LoRA 是 base 线性层上的低秩增量

先固定 base 权重，再按请求选择一个低秩增量。不同 adapter 不能只因名称相同就共用：base revision、输入/输出维度、rank、缩放和权重版本都是接口的一部分。


In [ ]:
base = ((1.0, 0.0), (0.0, 1.0))  # 执行本行的状态、计算或校验逻辑。
adapters = {"legal@1": {"tenant": "legal", "base": "b1", "a": ((1.0, 0.0),), "b": ((0.5,), (0.0,)), "alpha": 1.0}, "support@1": {"tenant": "support", "base": "b1", "a": ((0.0, 1.0),), "b": ((0.0,), (0.5,)), "alpha": 1.0}}  # 执行本行的状态、计算或校验逻辑。
assert len(adapters) == 2  # 执行本行的状态、计算或校验逻辑。
assert all(item["base"] == "b1" for item in adapters.values())  # 执行本行的状态、计算或校验逻辑。
assert all(len(item["a"]) == 1 for item in adapters.values())  # 执行本行的状态、计算或校验逻辑。

## 2. 从零计算 base + B(Ax) 的 forward

这里不用深度学习框架的 adapter 层：先算 A x，再算 B(Ax)，最后乘缩放并加回 base 输出。真实 kernel 会把请求按 rank/layout 分组，但数学与版本约束不能丢。


In [ ]:
def matvec(matrix, vector):  # 执行本行的状态、计算或校验逻辑。
    return tuple(sum(weight * value for weight, value in zip(row, vector)) for row in matrix)  # 执行本行的状态、计算或校验逻辑。
def forward(vector, adapter):  # 执行本行的状态、计算或校验逻辑。
    base_output = matvec(base, vector)  # 执行本行的状态、计算或校验逻辑。
    hidden = matvec(adapter["a"], vector)  # 执行本行的状态、计算或校验逻辑。
    delta = matvec(adapter["b"], hidden)  # 执行本行的状态、计算或校验逻辑。
    rank = len(adapter["a"])  # 执行本行的状态、计算或校验逻辑。
    return tuple(left + adapter["alpha"] * right / rank for left, right in zip(base_output, delta))  # 执行本行的状态、计算或校验逻辑。
legal_output = forward((2.0, 3.0), adapters["legal@1"])  # 执行本行的状态、计算或校验逻辑。
assert legal_output == (3.0, 3.0)  # 执行本行的状态、计算或校验逻辑。
assert forward((2.0, 3.0), adapters["support@1"]) == (2.0, 4.5)  # 执行本行的状态、计算或校验逻辑。

## 3. 授权与兼容性先于加载

请求带 tenant 并不自动获得任何 adapter。服务层应先检查 adapter 存在、tenant、base revision 和 shape；失败不能退化为“随便用 base”，否则会把业务错误伪装成正常回答。


In [ ]:
def authorize(request, registry):  # 执行本行的状态、计算或校验逻辑。
    adapter = registry.get(request["adapter"])  # 执行本行的状态、计算或校验逻辑。
    return adapter is not None and adapter["tenant"] == request["tenant"] and adapter["base"] == request["base"]  # 执行本行的状态、计算或校验逻辑。
good_request = {"tenant": "legal", "adapter": "legal@1", "base": "b1"}  # 执行本行的状态、计算或校验逻辑。
assert authorize(good_request, adapters)  # 执行本行的状态、计算或校验逻辑。
assert not authorize({**good_request, "tenant": "support"}, adapters)  # 执行本行的状态、计算或校验逻辑。
assert not authorize({**good_request, "base": "b2"}, adapters)  # 执行本行的状态、计算或校验逻辑。

## 4. 内存池只驱逐未被 in-flight 请求引用的版本

LRU 只是一种候选排序；安全条件是 refcount 为零。请求开始时 pin adapter version，结束时 release，池满且没有可驱逐项则排队/拒绝，而不是回收仍在执行的权重。


In [ ]:
pool = {"legal@1": {"ref": 1, "last": 4}, "support@1": {"ref": 0, "last": 2}}  # 执行本行的状态、计算或校验逻辑。
def evictable(pool_value):  # 执行本行的状态、计算或校验逻辑。
    choices = [(item["last"], name) for name, item in pool_value.items() if item["ref"] == 0]  # 执行本行的状态、计算或校验逻辑。
    return min(choices)[1] if choices else None  # 执行本行的状态、计算或校验逻辑。
assert evictable(pool) == "support@1"  # 执行本行的状态、计算或校验逻辑。
assert evictable({"only": {"ref": 1, "last": 1}}) is None  # 执行本行的状态、计算或校验逻辑。
assert pool["legal@1"]["ref"] == 1  # 执行本行的状态、计算或校验逻辑。

## 5. batching 以兼容签名分组，而不是混算不同 rank

连续批处理可以将同一 base、rank、dtype 和 adapter layout 的请求放进同一 kernel。不同 adapter 权重可以在 grouped GEMM 中并行，但任何 shape/版本错配都要在调度前显式拆组或拒绝。


In [ ]:
requests = [{"id": "r1", "adapter": "legal@1", "rank": 1, "dtype": "fp16"}, {"id": "r2", "adapter": "support@1", "rank": 1, "dtype": "fp16"}, {"id": "r3", "adapter": "legal@1", "rank": 2, "dtype": "fp16"}]  # 执行本行的状态、计算或校验逻辑。
def signature(request):  # 执行本行的状态、计算或校验逻辑。
    return request["rank"], request["dtype"]  # 执行本行的状态、计算或校验逻辑。
groups = {}  # 执行本行的状态、计算或校验逻辑。
for request in requests:  # 执行本行的状态、计算或校验逻辑。
    groups.setdefault(signature(request), []).append(request["id"])  # 执行本行的状态、计算或校验逻辑。
assert groups[(1, "fp16")] == ["r1", "r2"]  # 执行本行的状态、计算或校验逻辑。
assert groups[(2, "fp16")] == ["r3"]  # 执行本行的状态、计算或校验逻辑。
assert len(groups) == 2  # 执行本行的状态、计算或校验逻辑。

## 6. 热更新创建新版本，旧版本延迟回收

覆盖 legal@1 会让同一 trace 在不同 token 使用不同权重。正确做法是发布 legal@2，新的请求显式选择它；legal@1 直到 refcount 归零才允许回收，并保留审计 trace。


In [ ]:
registry = {**adapters, "legal@2": {**adapters["legal@1"], "alpha": 2.0}}  # 执行本行的状态、计算或校验逻辑。
def choose_version(adapter_name, registry_value):  # 执行本行的状态、计算或校验逻辑。
    versions = sorted(name for name in registry_value if name.startswith(adapter_name + "@"))  # 执行本行的状态、计算或校验逻辑。
    return versions[-1]  # 执行本行的状态、计算或校验逻辑。
assert choose_version("legal", registry) == "legal@2"  # 执行本行的状态、计算或校验逻辑。
assert forward((2.0, 3.0), registry["legal@2"]) == (4.0, 3.0)  # 执行本行的状态、计算或校验逻辑。
assert registry["legal@1"]["alpha"] == 1.0  # 执行本行的状态、计算或校验逻辑。

## 7. 配额、观测与失败降级

指标要按 adapter version 报载入命中、排队、淘汰、refcount、显存、batch size 与质量回归。没有授权、base 不兼容、池满无安全 victim 时，返回显式错误或排队；不可以暗中换成另一个租户的 adapter。


In [ ]:
def admit(request, registry_value, pool_value):  # 执行本行的状态、计算或校验逻辑。
    if not authorize(request, registry_value):  # 执行本行的状态、计算或校验逻辑。
        return "reject_auth_or_base"  # 执行本行的状态、计算或校验逻辑。
    return "run" if request["adapter"] in pool_value or evictable(pool_value) is not None else "queue"  # 执行本行的状态、计算或校验逻辑。
assert admit(good_request, registry, pool) == "run"  # 执行本行的状态、计算或校验逻辑。
assert admit({**good_request, "tenant": "other"}, registry, pool) == "reject_auth_or_base"  # 执行本行的状态、计算或校验逻辑。
assert admit(good_request, registry, {"x": {"ref": 1, "last": 1}}) == "queue"  # 执行本行的状态、计算或校验逻辑。

## 8. 端到端验收用正确输出与隔离性共同验证

小样例中不同 adapter 必须产生可解释的不同 delta，且任意请求不会借到其他租户版本。生产还需压测 adapter churn、rank 分布和 KV/adapter 竞争，不能把单 adapter benchmark 当多租户结论。


In [ ]:
assert legal_output != forward((2.0, 3.0), adapters["support@1"])  # 执行本行的状态、计算或校验逻辑。
assert good_request["adapter"] in pool  # 执行本行的状态、计算或校验逻辑。
assert evictable(pool) != "legal@1"  # 执行本行的状态、计算或校验逻辑。
assert all("@" in name for name in registry)  # 执行本行的状态、计算或校验逻辑。

## 面试收束

面试主线是：base+低秩 delta 的数学、请求绑定的版本/租户/base 合同、refcount 优先于 LRU、兼容 batch 分组、不可变发布与可观测回退。S-LoRA 的统一分页是性能实现，不能替代授权与 in-flight 一致性。
